# Test-set bootstrap intervals and paired comparisons

Seed-to-seed spread only measures training randomness. It misses a second source of noise: the test set is a finite sample, and some classes are tiny (Worms has 34 test rows). The **bootstrap** captures it: the 408,468 test flows are resampled with replacement 1,000 times and every metric is recomputed on each resample.

- For each condition, the score is the **mean over its seeds** of macro F1 on the resample. The 95% interval is the 2.5th to 97.5th percentile over the 1,000 resamples.
- For a **difference** between two conditions, both are scored on the *same* resamples (paired bootstrap), so the interval shows whether the difference is distinguishable from zero given test-set noise.
- Every score is also given **without Worms** (macro F1 over the other 9 classes), because Worms alone can move macro F1 by several hundredths.

Inputs: the saved test predictions of the design-controls notebook (LightGBM variants, 3 seeds), of the GNN density notebook (GNN with square-root class weights, 5 seeds) and of the GNN weights notebook (GNN with full balanced class weights, 5 seeds). No model is retrained here.

In [ ]:
import glob

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATASET_PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
B = 1000
find = lambda name: glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]

test_df = pd.read_parquet(f"{DATASET_PATH}/test.parquet", columns=["label"])
class_names = list(joblib.load(f"{DATASET_PATH}/preprocessing.pkl")["label_encoder"].classes_)
y = test_df["label"].to_numpy().astype("int64")
n, K = len(y), len(class_names)
worms = class_names.index("Worms")

preds = pd.concat([pd.read_parquet(find("design_controls_test_pred.parquet")).loc[test_df.index],
                   pd.read_parquet(find("gnn_density_test_pred.parquet")).loc[test_df.index],
                   pd.read_parquet(find("gnn_weights_test_pred.parquet")).loc[test_df.index]], axis=1)
conditions = {}
for col in preds.columns:
    name, seed = col.rsplit("__", 1)
    conditions.setdefault(name, []).append(col)
print({k: len(v) for k, v in conditions.items()})

## 1. Bootstrap machinery

A resample is a vector of counts (how often each test row is drawn). Weighted counting of the label/prediction pairs gives a confusion matrix, and F1 per class follows from it.

In [ ]:
codes = {col: y * K + preds[col].to_numpy().astype("int64") for col in preds.columns}


def macro_f1s(col, w):
    cm = np.bincount(codes[col], weights=w, minlength=K * K).reshape(K, K)
    tp, support, predicted = np.diag(cm), cm.sum(1), cm.sum(0)
    f1 = np.where(support + predicted > 0, 2 * tp / np.maximum(support + predicted, 1e-12), 0.0)
    return f1.mean(), np.delete(f1, worms).mean()


rng = np.random.default_rng(0)
score = {c: np.zeros((B, 2)) for c in conditions}      # [:, 0] macro F1, [:, 1] macro F1 without Worms
for b in range(B):
    w = np.bincount(rng.integers(0, n, n), minlength=n).astype("float64")   # how often each test row is drawn
    for c, cols in conditions.items():
        score[c][b] = np.mean([macro_f1s(col, w) for col in cols], axis=0)

full = np.ones(n)
point = {c: np.mean([macro_f1s(col, full) for col in cols], axis=0) for c, cols in conditions.items()}
ci = lambda a: np.percentile(a, [2.5, 97.5])

## 2. Score of every condition with 95% interval

In [ ]:
rows = []
for c in conditions:
    lo, hi = ci(score[c][:, 0])
    lo9, hi9 = ci(score[c][:, 1])
    rows.append({"condition": c, "seeds": len(conditions[c]), "macro F1": point[c][0], "95% CI": f"{lo:.3f} - {hi:.3f}",
                 "macro F1, no Worms": point[c][1], "95% CI (no Worms)": f"{lo9:.3f} - {hi9:.3f}"})
table = pd.DataFrame(rows).set_index("condition")
print(table.round(3).to_string())

## 3. Paired differences

`A - B`: positive means A is better. The interval is the paired 95% bootstrap interval; "excludes 0" means the difference is larger than test-set noise (it does not by itself say the model would win on another split).

In [ ]:
comparisons = [
    ("two-stage, weighted, original", "single-stage, weighted, original", "two-stage design (both weighted)"),
    ("single-stage, weighted, original", "single-stage, unweighted, original", "class weights (single-stage)"),
    ("two-stage, weighted, original", "two-stage, weighted, + SMOTE rows", "no SMOTE rows (two-stage)"),
    ("single-stage, weighted, original", "single-stage, weighted, + SMOTE rows", "no SMOTE rows (single-stage)"),
    ("real graph, matched density", "no graph", "GNN: real graph vs no graph"),
    ("real graph, matched density", "shuffled edges, matched density", "GNN: real graph vs shuffled edges"),
    ("real graph, matched density", "real graph, full train graph", "GNN: matched density vs denser training graph"),
    ("real graph, full train graph", "no graph", "GNN: denser training graph vs no graph"),
    ("two-stage, weighted, original", "real graph, matched density", "best tree model vs GNN"),
    ("single-stage, weighted, original", "no graph", "single-stage tree vs GNN without graph"),
    ("real graph, matched density, full class weights", "no graph, full class weights", "GNN full weights: real graph vs no graph"),
    ("real graph, matched density, full class weights", "real graph, matched density", "GNN: full vs square-root class weights"),
    ("single-stage, weighted, original", "no graph, full class weights", "single-stage tree vs GNN without graph (both full weights)"),
    ("two-stage, weighted, original", "real graph, matched density, full class weights", "best tree model vs GNN (full weights)"),
]
rows = []
for a, b, label in comparisons:
    d = score[a] - score[b]
    d0 = point[a] - point[b]
    lo, hi = ci(d[:, 0])
    lo9, hi9 = ci(d[:, 1])
    rows.append({"comparison": label, "difference": d0[0], "95% CI": f"{lo:+.3f} to {hi:+.3f}", "excludes 0": bool(lo > 0 or hi < 0),
                 "diff, no Worms": d0[1], "95% CI (no Worms)": f"{lo9:+.3f} to {hi9:+.3f}", "excludes 0 (no Worms)": bool(lo9 > 0 or hi9 < 0)})
diffs = pd.DataFrame(rows).set_index("comparison")
print(diffs.round(3).to_string())

## 4. Figures

In [ ]:
order = list(conditions)
fig, ax = plt.subplots(figsize=(10, 0.45 * len(order) + 1.5))
for i, c in enumerate(order[::-1]):
    lo, hi = ci(score[c][:, 0])
    ax.errorbar(point[c][0], i, xerr=[[point[c][0] - lo], [hi - point[c][0]]], fmt="o", color="steelblue", capsize=4)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order[::-1])
ax.set_xlabel("test macro F1 (mean over seeds, 95% bootstrap interval)")
ax.set_title("Every condition with its test-set bootstrap interval")
plt.tight_layout()
plt.savefig("/kaggle/working/fig_01.png", dpi=130, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 0.5 * len(comparisons) + 1.5), sharey=True)
for ax, k, title in [(axes[0], 0, "all 10 classes"), (axes[1], 1, "without Worms (9 classes)")]:
    for i, (a, b, label) in enumerate(comparisons[::-1]):
        d = score[a][:, k] - score[b][:, k]
        lo, hi = ci(d)
        est = point[a][k] - point[b][k]
        ax.errorbar(est, i, xerr=[[est - lo], [hi - est]], fmt="o", color="darkorange" if (lo > 0 or hi < 0) else "gray", capsize=4)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_yticks(range(len(comparisons)))
    ax.set_yticklabels([c[2] for c in comparisons[::-1]])
    ax.set_xlabel("difference in macro F1 (A - B)")
    ax.set_title(f"Paired differences, {title}\n(orange = interval excludes 0)")
plt.tight_layout()
plt.savefig("/kaggle/working/fig_02.png", dpi=130, bbox_inches="tight")
plt.show()

table.to_csv("/kaggle/working/bootstrap_scores.csv")
diffs.to_csv("/kaggle/working/bootstrap_differences.csv")